# Long builds

Run the one code cell below. It runs the chunk build, then the holdout rebuild, then the duplicate check, in one kernel session, so the kernel stays busy for the whole run. A heartbeat line prints every minute.

If the run stops, run the cell again: the chunk build skips finished chunks. The holdout and duplicate check start over.

In [ ]:
import subprocess, threading, time

PY = "/home/sagemaker-user/venv-dsdk/bin/python"

def run(script):
    t0 = time.time()
    proc = subprocess.Popen([PY, script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stop = threading.Event()

    def heartbeat():
        while not stop.wait(60):
            print(f"[heartbeat] {script} running {(time.time() - t0) / 60:.0f} min", flush=True)

    threading.Thread(target=heartbeat, daemon=True).start()
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    stop.set()
    print(f"exit={rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    return rc

for script in [
    "/home/sagemaker-user/pipe/chunk_build.py",
    "/home/sagemaker-user/pipe/holdout_build.py",
    "/home/sagemaker-user/dup_check/dup_check.py",
]:
    if run(script) != 0:
        print(f"stopped: {script} failed; fix it and run this cell again", flush=True)
        break